In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

# علشان الرسومات تظهر بشكل أوضح
plt.style.use("ggplot")

# عرض جميع الأعمدة
pd.set_option("display.max_columns", None)

In [ ]:
train = pd.read_csv("Train.csv")
test = pd.read_csv("Test.csv")

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
cat_cols = X_train.select_dtypes(include="object").columns

num_cols = X_train.select_dtypes(
    include=["int64","float64"]
).columns
print(cat_cols)
print(num_cols)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ]
)

In [ ]:
train.head()

In [ ]:
X_train.dtypes

**Models**

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

lr_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]
)

lr_model.fit(X_train, y_train)

lr_pred = lr_model.predict(X_val)

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

print("R2:", r2_score(y_val, lr_pred))
print("MAE:", mean_absolute_error(y_val, lr_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_val, lr_pred)))

In [ ]:
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

xgb_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model",
         XGBRegressor(
             n_estimators=1000,
             learning_rate=0.03,
             max_depth=8,
             subsample=0.8,
             colsample_bytree=0.8,
             random_state=42
         ))
    ]
)

In [ ]:
xgb_model.fit(X_train, y_train)

In [ ]:
y_pred = xgb_model.predict(X_val)

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

r2 = r2_score(y_val, y_pred)
mae = mean_absolute_error(y_val, y_pred)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))

print("R2:", r2)
print("MAE:", mae)
print("RMSE:", rmse)

Hyperparameter Tuning

In [ ]:
param_grid_xgb = {
    "model__n_estimators": [500, 1000],
    "model__max_depth": [6, 8],
    "model__learning_rate": [0.03, 0.05]
}

In [ ]:
from sklearn.model_selection import GridSearchCV

grid_xgb = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid_xgb,
    cv=3,
    scoring="r2",
    n_jobs=-1
)

In [ ]:
grid_xgb.fit(X_train, y_train)

In [ ]:
print("Best Parameters:", grid_xgb.best_params_)
print("Best CV Score:", grid_xgb.best_score_)

In [ ]:
best_xgb = grid_xgb.best_estimator_

xgb_pred = best_xgb.predict(X_val)

print("R2:", r2_score(y_val, xgb_pred))
print("MAE:", mean_absolute_error(y_val, xgb_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_val, xgb_pred)))

Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(random_state=42))
])

In [ ]:
rf_model.fit(X_train, y_train)

In [ ]:
rf_pred = rf_model.predict(X_val)

print("R2:", r2_score(y_val, rf_pred))
print("MAE:", mean_absolute_error(y_val, rf_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_val, rf_pred)))

Hyperparameter Tuning For Random Forest

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [10, None],
    "model__min_samples_split": [2, 5]
}

random_rf = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=param_dist,
    n_iter=4,
    cv=3,
    scoring="r2",
    random_state=42,
    n_jobs=-1
)

random_rf.fit(X_train, y_train)

print(random_rf.best_params_)
print(random_rf.best_score_)

In [ ]:
best_rf = random_rf.best_estimator_

rf_pred = best_rf.predict(X_val)

print("R2:", r2_score(y_val, rf_pred))
print("MAE:", mean_absolute_error(y_val, rf_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_val, rf_pred)))

Model Comparsion

In [ ]:
import pandas as pd

results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest",
        "XGBoost"
    ],
    "R2": [
        0.747845502165269,
        0.9324571731632378,
        0.9404846429824829
    ],
    "MAE": [
        762.0176931347157,
        320.71408847595245,
        308.46014404296875
    ],
    "RMSE": [
        1003.502791786594,
        519.3675863358295,
        487.52822033703853
    ]
})

results.sort_values("R2", ascending=False)